In [0]:
from pyspark.sql import functions as F

In [0]:
t = spark.table("workspace.default.taxi_raw")

In [0]:
t.createOrReplaceTempView("taxi_raw")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW taxi_2025 AS
SELECT *
FROM taxi_raw
WHERE YEAR(tpep_pickup_datetime) = 2025
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime IS NOT NULL
  AND tpep_dropoff_datetime > tpep_pickup_datetime;

In [0]:
%sql
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickups,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoffs
FROM taxi_2025;

total_trips,null_pickups,null_dropoffs
48176269,0,0


In [0]:
%sql
SELECT COUNT(*) AS total_trips
FROM taxi_2025;

total_trips
48176269


Trips By Month

In [0]:
%sql
SELECT
    MONTH(tpep_pickup_datetime) AS month,
    COUNT(*) AS total_trips
FROM taxi_2025
GROUP BY MONTH(tpep_pickup_datetime)
ORDER BY month;

month,total_trips
1,3473183
2,3572378
3,4122949
4,3935799
5,4527572
6,4254366
7,3842907
8,3526247
9,4193840
10,4360740


Trips By Day of the Week

In [0]:
%sql
SELECT
    DAYOFWEEK(tpep_pickup_datetime) AS day_number,
    DATE_FORMAT(tpep_pickup_datetime, 'EEEE') AS day_of_week,
    COUNT(*) AS total_trips
FROM taxi_2025
GROUP BY
    DAYOFWEEK(tpep_pickup_datetime),
    DATE_FORMAT(tpep_pickup_datetime, 'EEEE')
ORDER BY day_number;

day_number,day_of_week,total_trips
1,Sunday,6380236
2,Monday,5880986
3,Tuesday,6680291
4,Wednesday,7208642
5,Thursday,7393379
6,Friday,7175788
7,Saturday,7456947


Busiest and least busiest hour 

In [0]:
%sql
SELECT
    HOUR(tpep_pickup_datetime) AS hour,
    COUNT(*) AS total_trips
FROM taxi_2025
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY hour;

hour,total_trips
0,1524881
1,1004414
2,661941
3,453162
4,358620
5,383682
6,752506
7,1368603
8,1847181
9,1973213


Busiest Hour

In [0]:
%sql
SELECT *
FROM (
    SELECT
        HOUR(tpep_pickup_datetime) AS hour,
        COUNT(*) AS total_trips
    FROM taxi_2025
    GROUP BY HOUR(tpep_pickup_datetime)
)
ORDER BY total_trips DESC
LIMIT 1;

hour,total_trips
18,3432279


Least busy hour

In [0]:
%sql
SELECT *
FROM (
    SELECT
        HOUR(tpep_pickup_datetime) AS hour,
        COUNT(*) AS total_trips
    FROM taxi_2025
    GROUP BY HOUR(tpep_pickup_datetime)
)
ORDER BY total_trips ASC
LIMIT 1;

hour,total_trips
4,358620


The Three busiest routes

In [0]:
%sql
SELECT
    PULocationID,
    DOLocationID,
    COUNT(*) AS total_trips
FROM taxi_2025
WHERE PULocationID IS NOT NULL
  AND DOLocationID IS NOT NULL
  AND PULocationID <> DOLocationID
GROUP BY
    PULocationID,
    DOLocationID
ORDER BY total_trips DESC
LIMIT 3;

PULocationID,DOLocationID,total_trips
237,236,296255
236,237,253994
161,237,140734


In [0]:
zones.createOrReplaceTempView("taxi_zones")

In [0]:
%sql
SELECT
    pu.Zone AS pickup_zone,
    pu.Borough AS pickup_borough,
    do.Zone AS dropoff_zone,
    do.Borough AS dropoff_borough,
    COUNT(*) AS total_trips
FROM taxi_2025 t
JOIN taxi_zones pu
    ON t.PULocationID = pu.LocationID
JOIN taxi_zones do
    ON t.DOLocationID = do.LocationID
WHERE t.PULocationID IS NOT NULL
  AND t.DOLocationID IS NOT NULL
  AND t.PULocationID <> t.DOLocationID
GROUP BY
    pu.Zone,
    pu.Borough,
    do.Zone,
    do.Borough
ORDER BY total_trips DESC
LIMIT 3;

pickup_zone,pickup_borough,dropoff_zone,dropoff_borough,total_trips
Upper East Side South,Manhattan,Upper East Side North,Manhattan,296255
Upper East Side North,Manhattan,Upper East Side South,Manhattan,253994
Midtown Center,Manhattan,Upper East Side South,Manhattan,140734
